In [2]:
"""
데이터 분석가 채용공고 지역별 분포 지도 시각화
사용 파일:
  - posting_analysis_table_최종.xlsx
  - skorea-municipalities-2018-geo.json
"""

import json
import pandas as pd
import folium
import branca.colormap as cm


# ── 상수 ──────────────────────────────────────────────────────────────────────

DATA_PATH   = r"C:\py_temp\중간프로젝트\posting_analysis_table_최종.xlsx"
GEO_PATH    = r"C:\py_temp\중간프로젝트\skorea-municipalities-2018-geo.json"
OUTPUT_PATH = r"C:\py_temp\중간프로젝트\채용공고_지역별_지도.html"

# 행정구역 코드 앞 2자리 → 시도명
# GeoJSON에는 구/시/군 이름만 있고 시도 정보가 없으므로,
# code 앞 2자리로 시도를 복원해 "서울 중구" / "인천 중구" 를 구분함
SIDO_CODE_MAP = {
    "11": "서울", "21": "부산", "22": "대구", "23": "인천",
    "24": "광주", "25": "대전", "26": "울산", "29": "세종",
    "31": "경기", "32": "강원", "33": "충북", "34": "충남",
    "35": "전북", "36": "전남", "37": "경북", "38": "경남", "39": "제주",
}

# 해외 시도 값 (필터링 제거 대상)
OVERSEAS_SIDOS = {"동경", "미국", "베트남", "일본", "헝가리"}


# ── 1. 데이터 로드 및 전처리 ───────────────────────────────────────────────────

def load_and_clean(path: str) -> pd.DataFrame:
    """엑셀 로드 → 해외/이상 데이터 제거 → region_key 생성"""
    df = pd.read_excel(path)

    # 해외 지역 제거
    df = df[~df["region_sido"].isin(OVERSEAS_SIDOS)].copy()

    # 시각화용_지역이 "시도 구/시" 형태가 아닌 행 제거 (예: "서울" 단독, 세종 도로명 등)
    df = df[df["시각화용_지역"].str.contains(" ", na=False)].copy()

    # "서울 강남구 일부텍스트" 같은 경우 대비해 앞 두 토큰만 사용
    df["region_key"] = df["시각화용_지역"].str.strip().str.split().str[:2].str.join(" ")

    return df


def aggregate_by_region(df: pd.DataFrame) -> pd.DataFrame:
    """지역별 공고 수, 평균 최종점수, 등급 분포 집계"""
    agg = df.groupby("region_key").agg(
        공고수=("posting_id", "count"),
        평균점수=("posting_final_score", "mean"),
    ).reset_index()
    agg["평균점수"] = agg["평균점수"].round(1)
    return agg


# ── 2. GeoJSON 전처리 ─────────────────────────────────────────────────────────

def load_and_enrich_geojson(path: str, agg: pd.DataFrame) -> dict:
    """
    GeoJSON 각 feature에 sido_gu 키(예: '서울 중구')와
    집계값(공고수, 평균점수)을 주입해 반환
    """
    with open(path, encoding="utf-8") as f:
        gj = json.load(f)

    count_map = dict(zip(agg["region_key"], agg["공고수"]))
    score_map = dict(zip(agg["region_key"], agg["평균점수"]))

    for feature in gj["features"]:
        code = feature["properties"]["code"]
        sido = SIDO_CODE_MAP.get(code[:2], "")
        gu   = feature["properties"]["name"]
        key  = f"{sido} {gu}"

        feature["properties"]["sido_gu"]  = key
        feature["properties"]["공고수"]    = count_map.get(key, 0)
        feature["properties"]["평균점수"]  = score_map.get(key, None)

    return gj


# ── 3. 지도 생성 ──────────────────────────────────────────────────────────────

def build_map(gj: dict, max_count: int) -> folium.Map:
    """choropleth 지도 생성"""

    m = folium.Map(
        location=[36.5, 127.8],
        zoom_start=7,
        tiles="CartoDB positron",
    )

    # 공고 수 기준 색상맵
    colormap = cm.LinearColormap(
        colors=["#f7fbff", "#c6dbef", "#6baed6", "#2171b5", "#08306b"],
        vmin=0,
        vmax=max_count,
        caption="채용공고 수",
    )
    colormap.add_to(m)

    def style_fn(feature):
        count = feature["properties"].get("공고수", 0)
        if count == 0:
            return {"fillColor": "#eeeeee", "color": "#cccccc",
                    "weight": 0.5, "fillOpacity": 0.3}
        return {"fillColor": colormap(count), "color": "#555555",
                "weight": 0.7, "fillOpacity": 0.75}

    def highlight_fn(feature):
        return {"fillOpacity": 0.95, "weight": 2, "color": "#333333"}

    tooltip = folium.GeoJsonTooltip(
        fields=["sido_gu", "공고수", "평균점수"],
        aliases=["지역", "공고 수", "평균 점수"],
        localize=True,
        sticky=True,
        style="""
            background-color: white;
            border: 1px solid #ccc;
            border-radius: 4px;
            padding: 8px;
            font-size: 13px;
            font-family: 'Malgun Gothic', sans-serif;
        """,
    )

    folium.GeoJson(
        gj,
        style_function=style_fn,
        highlight_function=highlight_fn,
        tooltip=tooltip,
    ).add_to(m)

    # 타이틀
    title_html = """
    <div style="
        position: fixed; top: 15px; left: 50%; transform: translateX(-50%);
        z-index: 1000; background: white;
        padding: 10px 24px; border-radius: 8px;
        box-shadow: 0 2px 8px rgba(0,0,0,0.2);
        font-family: 'Malgun Gothic', sans-serif;
        font-size: 16px; font-weight: bold; color: #222;
    ">
        📊 데이터 분석가 채용공고 지역별 분포
    </div>
    """
    m.get_root().html.add_child(folium.Element(title_html))

    return m


# ── 4. 실행 ───────────────────────────────────────────────────────────────────

def main():
    print("▶ 데이터 로드 중...")
    df = load_and_clean(DATA_PATH)
    print(f"  국내 공고: {len(df)}건")

    agg = aggregate_by_region(df)
    print(f"  집계 지역: {len(agg)}개")

    print("▶ GeoJSON 전처리 중...")
    gj = load_and_enrich_geojson(GEO_PATH, agg)

    print("▶ 지도 생성 중...")
    m = build_map(gj, max_count=int(agg["공고수"].max()))

    m.save(OUTPUT_PATH)
    print(f"✅ 저장 완료: {OUTPUT_PATH}")


if __name__ == "__main__":
    main()


▶ 데이터 로드 중...
  국내 공고: 875건
  집계 지역: 101개
▶ GeoJSON 전처리 중...
▶ 지도 생성 중...
✅ 저장 완료: C:\py_temp\중간프로젝트\채용공고_지역별_지도.html
